## PA3606 UNIT 2: Climate Physics
# Computer Workshop 2: Investigating Atmospheric Lapse Rates & Stability

#### Purpose of these workshops
For Unit 2 of PA3606 we will have 2 computer workshops which you should think of as guided lectures. There will be no expectation for you to write lots of code, rather the objectives are to:
1. Introduce you to the Jupyter environment
2. learn how to put together basic workflow to analyse data
3. Gain experience working with blocks of python code
4. Use knowledge from our lectures to interpret results 

#### Python Libraries
For all our computer workshsops we will work with four python libraries:
1. **NumPy** (https://numpy.org/): _"NumPy is the fundamental package for scientific computing in Python. It is a Python library that provides a multidimensional array object, various derived objects (such as masked arrays and matrices), and an assortment of routines for fast operations on arrays, including mathematical, logical, shape manipulation, sorting, selecting, I/O, discrete Fourier transforms, basic linear algebra, basic statistical operations, random simulation and much more"_. We will use this for most of our math functions
2. **SciPy** (https://scipy.org/): _"SciPy is a collection of mathematical algorithms and convenience functions built on NumPy . It adds significant power to Python by providing the user with high-level commands and classes for manipulating and visualizing data"_.This is the other library we wil use for statistical/algorithmic functions.
3. **NetCDF4** (https://unidata.github.io/netcdf4-python/): The Network Common Data Form (NetCDF) is specific scientific data format used in climate and earth observation sciences. Each variable is stored in a compressed format along with accompanying meta data. The majority of data you will use in this course wil be stored in this format.
4. **Matplotlib** (https://matplotlib.org/): _"Matplotlib is a comprehensive library for creating static, animated, and interactive visualizations in Python. Matplotlib makes easy things easy and hard things possible"_. We will use this library to visualise the contents of data files and the results from the analyses we perform on these data.


# Atmospheric Lapse Rate
### Last lecture we looked at how the structures of planetary atmospheres are defined by temperature gradients, or atmospheric lapse rates. The first concept we looked at that related to structure to temperature gradients, or lapse rates:

![lapse rates](img/lapse_rate_example.png)

### * When the temperature increases with altitude, the atmosphere is stable
### * When the temperature decreases with altitude, the atmosphere is less stable

### We also came across our first definition based off the first law of thermal dynamics

![dry lapse rate](img/dryAdiabaticLapseRate.png)

***
# Exercise 1: Investigating the lapse rate over Lindenberg, Germany
### For this first exercise, we examine the tropospheric lapse rate between 2008 and 2016 using radiosonde data from weather balloons launched over Lindenberg, Germany. The dataset we will use comes from the Global Climate Observing System (GCOS) Reference Upper-Air Network (GRUAN) database (https://www.gruan.org/), which contains measurements from approximately 30 sites worldwide. GRUAN will impose a set of standard operating procedures on all sites that wish to qualify for the network (~1000 radionde sites globally). This ensures calibration consistency across the network, standardised measurement characterisation, and the production of uncertainties. Therefore, GRUAN observations are considered to be 'climate quality' data.  

![gruan](img/gruan_sites.png)

### Lets begin by reading in some temperature profile data
First we'll load our Python libraries we need:

In [ ]:
import netCDF4                  # used for reading in the data
import matplotlib.pyplot as plt # used for plotting the data
import numpy as np              # used for math functions and creating arrays etc
from scipy import stats         # stats module for linear regression

Next we want to read in the radiosonde data from Lindenberg. The steps we follow are:
1. define a variable called ``` filename ``` that tells the computer where to find the data
2. read in the data using the <span style="color:blue">**netCDF4**</span>
3. next we want to take a look at what is inside the file
4. extract the data and close the file

In [ ]:
# 1. define filename
filename = "data/gruan_temperature_profiles_lindenberg_2008_to_2016.nc"

# 2. read in the data to memory. We use a unit test approach (good practice) when doing this, the 'try' command
# will first attempt to execute the command witten underneath whichin our case is to read the file into memory,
# if it fails it raises a specific error which tells youthereis an issue with the command
try:
    nc = netCDF4.Dataset(filename)
except IOError:
    raise
# print the global meta data for the file we have just read in
nc

from the global meta data we can see that the data inside:
* runs from 2008-2016
* has two dimsions ```dimensions(sizes): time(11233), level(80)```
* has three variables; i) altitude, ii) time, and ii) T

lets look atthe variables in a bit more detail:

In [ ]:
# print the variable information to the screen
nc.variables

from this we can see that the details of each variable are:

| variable | dimensions              | units                             |
|----------|-------------------------|-----------------------------------|
| altitude | level (80)              | m                                 |
| time     | time (11233)            | seconds since 2008-01-01 00:00:00 |
| T        | level x time (80, 11233)| K                                 |

In [ ]:
# 4. Extract the data and close the file

# alttiude (m)
alt = nc['altitude'][:] # here we index the variable in the object called nc and select all the data with [:]

# temperature (K)
T = nc['T'][:]

# time - here we extract the time in seconds since 1st Januray 2008. The netCDF4 librayy has a function called 
# num2date method which takes the date and time information held in seconds and converts it into a datetime object.
# We don't use it here, but could if we wanted to have year,month, day, hour, minute ,second information for each sounding.
time = nc['time'][:]

# close the file
nc.close()

#### Next lets make a simple visualisation of the data

In [ ]:
# set the figure size or dimensions
plt.figure(figsize=(15,3))
# plot our 2D temperature data usingthe time and alt arrays to index the different dimensions
# convert altitudes fromm to km (*1e-3) and temperatures from K to degrees C
plt.pcolormesh(time, alt*1e-3, T-273.15,vmin=-30, vmax=30,cmap=plt.get_cmap('coolwarm',12))
# fix the extentof the y axis between 0 and 8 km
plt.ylim(0,8)
# add the label for the y axis
plt.ylabel("Altitude (km)")
# add the label for the x axis
plt.xlabel("Time (seconds since 2008-01-01 00:00:00)")
# add a colour bar and change the end to traiangles to indicatethat the range goes beyond the fixed range (+/- 30)
plt.colorbar(extend='both')
# add a title
plt.title(r"Gruan Atmospheric Temperature Soundings ($^{\circ}$C)")

#### Next we want to calculate the environemntal lapse for each of the 11,233 profiles. To do this we write a function into which we can pass the temperatures and altitudes

In [ ]:
def calc_lapse_rate(Tprof, zprof):
    """ calculate the environemntal lapse rate for a set of profiles
    inputs Tprof -> atmospheric temperature profiles (K)
           zprof -> altitude profile (km)

    outputs laspe -> environmental lapse rate (K/km)
    """
    # define an array to hold the lapse rates based on the number of profiles in the
    # array Tprof (e.g. dimension 1)
    nprofs = Tprof.shape[1]
    lapse = np.full(nprofs,np.nan)
    # loop over each profile and calculate the environmental lapse rate
    # Note: we are assuming a constant rate between 1 and 8 km
    for tt in range(lapse.size):
        # select the profile and mask any missing values (e.g. NaNs)
        x = np.ma.masked_invalid(Tprof[:,tt])
        # apply the mak to the altitude values
        y = np.ma.masked_array(zprof,mask=x.mask)
        # use the compressed() method to remove all the bad values 
        x, y = x.compressed(), y.compressed()
        # find the points above 1 km in altitude using the where function
        find = np.where(y > 1)
        # filter the profile and altitudes with the indexs from the where statement 
        x, y = x[find], y[find]
        # calculate the lapse rate and store in the correct position in the output array
        lapse[tt] = -1*((x[-1]-x[0])/(y[-1]-y[0]))
    # return the lapse rates
    return lapse

#### Calculate lapse rate, remembering to convert the units of altitude from m to km

In [ ]:
lapse_rate = calc_lapse_rate(T, alt*1e-3) # *1e-3 -> m to km  

#### after we calculate the lapse rate what we want to do is look at the spread in the data. To do this we will calculate a Probability Density Function (PDF) using the python function defined below. **Note:** This function assumes our results have a Gaussian distribution.   

In [ ]:
def calculate_probDensFunc(yvals, ymin, ymax,nbins=100):
    """ calculate a PDF from the array yvals using scipy.stats norm module
    inputs: yvals -> 1d array of values to be used in PDF calculation
            ymin  -> minimum value for range of yvals PDF is calculated
            ymax  -> maximum value for range of yvals PDF is calculated
            nbins -> number of bins for which PDF is calculated between ymin  and ymax
            
    outputs: xvals -> array of values over which the PDF was calculated (defined by ymin, ymax,nbins)
             PDF   -> array containing PDF values
    """
    # define xvals
    xvals = np.linspace(ymin,ymax,nbins)

    # calculate mean and standard deviation of yvals
    mu = np.mean(yvals)
    std = np.std(yvals)

    # define an empty array to hold PDF values. Here we use the size method to tell the code how many elements 
    # are in this new array and fill each entry with a default value NaN (Not a Number)
    PDF = np.full(xvals.size, np.nan)

    # loop over each value in x and calculate the corresponding PDF value. Because xvals is an array merans in 
    # Python it is iterable (i.e. we can loop over the contents) and by wrapping it in the enumerate function
    # we also get the index of the value (e.g. the firts value in xvals could be -2, therefore x=-2 and ii=0).
    for ii, x in enumerate(xvals):
        PDF[ii] = stats.norm.pdf(x, loc=mu, scale=std)

    # return the results
    return PDF, xvals

In [ ]:
lapse_pdf,xvals = calculate_probDensFunc(lapse_rate, 4, 9,nbins=100)

#### Finally, lets plot our results:

In [ ]:
# define the figure size
plt.figure(figsize=(6,5))
# calculate the mean environmental lapse rate 
mu = np.mean(lapse_rate)
h = plt.hist(lapse_rate,bins=100,density=True,alpha=0.3,color='darkorange',label='raw data')
# plot the PDF
plt.plot(xvals, lapse_pdf,'-',lw=3,color="#00A7B5",label=r"$\mu$ = "+f"{mu:0.2f} km/K")
# set x and y axis limits
plt.xlim(4,9)
plt.ylim(0,0.8)
# add a legend to display the mean lapse rate value
plt.legend(loc=1,fontsize=12) # loc = location, location 1 = top right
# add labels to both x and y axes
plt.ylabel("PDF",fontsize=14)
plt.xlabel(r"$-\frac{dT}{dz}$ [km/K]",fontsize=14) # the r before the " allows us to place LaTeX syntax intot he string
# add a title
plt.title("Enviromental Lapse Rate Calculated over\n Lindenberg (2008-2016)",fontsize=14) # the \n put the rest of the string on a new line

### What we can infer from this plot?
***

# Exercise 2: Investigating atmospheric stability

### The results from exercise 1 show us that the average observed lapse rate is not -9.8 km/K, but rather closer to 6.3 km/K. Using the idea of different lapse rates, we can begin to rate the current temperature gradient to atmospheric stability. 

![moist lapse rate](img/moistLapseRate.png) 

### so we can see that the result we got over Lindenberg from the GRUAN data is closer to the $\Gamma_{sat}$ lapse rate than the $\Gamma_{dry}$ lapse rate. From the measured atmospheric lapse rate ($\Gamma_{Atm}$) we can make an inference about the atmopsheric stability:

![conditional stability](img/condStability.png)

### Therefore, we can see that the mean observed $\Gamma_{Atm}$ lapse rate from GRUAN over Lindenberg indicates that the atmosphere is stable. However, if we look at the variability, we can also see that some soundings show conditionally unstable atmospheres. In this exercise, we are going to look at three example profiles from radiosonde measurements made at the Met Office observation site at Nottingham, Wollaton (WMO id 03354). The station is located 117m above mean sea level in Nottinghamshire, East UK (https://www.youtube.com/watch?v=Su8Bhp3XEY4). The details for each launch are stored in the following files:

![radisonde launch](img/balloonLaunchNottingham.png)
### First lets define the path to the files we are going to examine:

In [ ]:
# define full paths to radiosnde soundings over Nottingham Watnall site
file1 = "data/Nottingham2020.txt"
file2 = "data/Nottingham2022.txt"
file3 = "data/Nottingham2024.txt"

you will notice the foramt for these files is ```.txt``` which means they are human readable. If ou double click on one of the files you can see it will open in a neighbouring tab:
![txt file contents example](img/file_contents_example.png)

from the file header we can see these files contain the following information:

| Variable | Long Name                          | Units       |
|----------|-----------------------------------|------------|
| PRES     | Pressure                          | hPa        |
| HGHT     | Height                            | m          |
| TEMP     | Temperature                       | °C         |
| DWPT     | Dew Point Temperature             | °C         |
| RELH     | Relative Humidity                 | %          |
| MIXR     | Mixing Ratio                      | g/kg       |
| DRCT     | Wind Direction                    | degrees    |
| SKNT     | Wind Speed                        | knots      |
| THTA     | Potential Temperature             | K          |
| THTE     | Equivalent Potential Temperature  | K          |
| THTV     | Virtual Potential Temperature     | K          |

### lets create a function to read in the file and extract the height and temperature profile information. For this function we will make use of the <span style="color:blue">np.loadtxt</span> method. If we want to know more about a method from a library in Python we can use the ```help``` command.
```python
    help(np.loadtxt)
    
Help on function loadtxt in module numpy:

loadtxt(fname, dtype=<class 'float'>, comments='#', delimiter=None, converters=None, skiprows=0, usecols=None, unpack=False, ndmin=0, encoding=None, max_rows=None, *, quotechar=None, like=None)
    Load data from a text file.
    
    Parameters
    ----------
    fname : file, str, pathlib.Path, list of str, generator
        File, filename, list, or generator to read.  If the filename
        extension is ``.gz`` or ``.bz2``, the file is first decompressed. Note
        that generators must return bytes or strings. The strings
        in a list or produced by a generator are treated as lines.
    dtype : data-type, optional
        Data-type of the resulting array; default: float.  If this is a
        structured data-type, the resulting array will be 1-dimensional, and
        each row will be interpreted as an element of the array.  In this
        case, the number of columns used must match the number of fields in
        the data-type.
        ....
```

In [ ]:
# uncomment the line below to see the full help file 
# help(np.loadtxt)

### Let's define our function to read in the data. The np.loadtxt function will return a 2D array from which we can index the two variables we want to look at: 

In [ ]:
def read_in_radiosonde_profile(filename):
    """ read in radiosonde text files from University of Wyoming archive.
    inputs: filename -> full path to file

    outputs: hght    -> radiosonde altitude profile (km)
             temp    -> radisonde temperature profile (K)
    """
    # read in data skipping the first 4 rows as this contains the header information 
    data = np.loadtxt(filename,skiprows=5)

    # extract altitude information and convert units
    HGHT = data[:,1]*1e-3 # m to km
    # extract the temperature profile information
    TEMP = data[:,2]
    # create a mask for any bad/missing data points
    bad = (HGHT == -999)|(TEMP == -999)
    # mask and remove bad data points using .compressed()
    hght = np.ma.masked_array(HGHT,mask=bad).compressed()
    temp = np.ma.masked_array(TEMP,mask=bad).compressed()

    return hght, temp

### Now we have the method to read in the data let read in all three profiles and store them as variables

In [ ]:
# read in filename 1 (2020)
alts1, tmp1 = read_in_radiosonde_profile(file1)
# read in filename 2 (2022)
alts2, tmp2 = read_in_radiosonde_profile(file2)
# read in filename 3 (2024)
alts3, tmp3 = read_in_radiosonde_profile(file3)

### We want to plot these results using a multipanel plot so we can compre them side-by-side. Therefore, we need to create a generic plotting finction which we can reuse for each profile

In [ ]:
def plot_temp_profile(alts, tmp):
    """ generic profile plotting code
    inputs: alts -> altitude profile (km)
            tmp  -> temperature profile (K)

    outputs: plot including moist and dry lapse rate adiabats
    """
    # first calculate the moist and dry lapse rate profiles
    T0 = tmp[0] # take the near-surface air temperature value
    # calculate the upper temperature value based on a constant moist lapse rate
    T1 = T0 -((alts[-1]-alts[0])*6.5)
    # calculate the upper temperature value based on a constant dry adiabatic lapse rate
    T2 = T0 -((alts[-1]-alts[0])*9.8)
    # plot lapse rate lines
    plt.plot([T0, T1], [0,alts[-1]],'--',color='k',label='Moist Lapse Rate')
    plt.plot([T0, T2], [0,alts[-1]],':', color='k',label='Dry Adiabatic Lapse Rate')
    # plot temperature profile
    plt.plot(tmp,alts,color='r', label='Atmospheric Temperature')
    # set limits for x and y axes
    plt.ylim(0,15)
    plt.xlim(-75,50)
    # add labels for x and y axes
    plt.ylabel("Altitude (km)")
    plt.xlabel(r"Temperature ($^{\circ}$C)")
    # add major grid lines
    plt.grid(True)
    # add a legend
    plt.legend()

### now we make our multipanel plot using ```plt.subplot()```, where the numbers in the parentheses represent the (number of rows, number of columns, plot number). We plot each profile using the <span style="color:blue">plot_temp_profile()</span> function above, adding a title as we go to distinquish between the different profiles

In [ ]:
# set the plot size
plt.figure(figsize=(13,4))

# first plot, filename 1 (2020)
plt.subplot(131)
plot_temp_profile(alts1, tmp1)
plt.title("Case A: Nottingham 2020")

# second plot, filename 2 (2022)
plt.subplot(132)
plot_temp_profile(alts2, tmp2)
plt.title("Case B: Nottingham 2022")

# third plot, filename 3 (2024)
plt.subplot(133)
plot_temp_profile(alts3, tmp3)
plt.title("Case C: Nottingham 2024")

# optimise white space around plots
plt.tight_layout()

##  what can we infer from these plots?
### follow this link https://forms.gle/N9TudyxA6hstXETh7 and enter your answers in the table

***
# **How is this information used by meteorologists?**

***
![atmos stability tephigram](img/tephigram_1.png)
***
![atmos stability tephigram](img/tephigram_2.png)
***
![atmos stability tephigram](img/cape_cin_1.png)
***
![atmos stability tephigram](img/cape_cin_2.png)

# Reviewing the previous examples for Nottingham

***
# Tephigram – 2020 June severe convective event
![atmos stability tephigram](img/tephigram_3.png)
***
# Tephigram – 2022 UK heatwave case study
![atmos stability tephigram](img/tephigram_4.png)
***
# Tephigram – 2024 December anticyclonic gloom
![atmos stability tephigram](img/tephigram_5.png)

***
# Final exercise: reading tephigrams
### The code below will produce tephigrams for 5 examples taken from sites outsideof the UK. First, we will load a Python script (tephi.py) that generates tephigrams. Note that we can load functions from other pieces of Python code in the same way we load Python modules:
```python
# now we are going to load a special library to plot the radiosonde data
from tephi import plot_tephigram, parse_sounding_file
```
### From tephi.py we are loading a function that we parse the file contents we need to our Jupyter session, and another which will create the plot.

In [ ]:
# now we are going to load a special library to plot the radiosonde data
from tephi import plot_tephigram, parse_sounding_file

In [ ]:
# define a list of file to loop over
sounding_files = ["data/texas_cold_outbreak_16022021.txt",
                 'data/newcastle_moore_tornado_21052013.txt',
                 'data/ohio_tornado_28052019.txt',
                 'data/warsaw_cold_outbreak_06022012.txt',
                 'data/zagreb_supercell_19072023.txt',
                ]

In [ ]:
# loop over each file and create a plot.
# NOTE: this will save a .png file, which you can open inside the Jupyter notebook
for sounding_file in sounding_files:
    location = sounding_file.split('/')[1].split('_')[0]
    print(f"Parsing sounding data from: {sounding_file}")
    pressure, temperature, dewpoint, wind_dir, wind_speed, launchtime = parse_sounding_file(sounding_file)
    print(f"Loaded {len(pressure)} levels from {pressure[0]:.0f} hPa to {pressure[-1]:.0f} hPa")

    fig, ax = plot_tephigram(
        pressure,
        temperature,
        dewpoint,
        wind_dir,
        wind_speed,
        location=location.capitalize(),
        date=launchtime.strftime("%d.%m.%Y"),
        time=launchtime.strftime("%H:%M UTC"),
        save_path=f"radiosonde_sounding_{location}_{launchtime:%Y%m%d_%H%M}_utc.png"
    )